# exp090_lateral_self_gr_match_pseudotail_probe train

Train-side ablation of same-horizontal-well prefix/evaluation GR match summary features on the fixed exp073 full replay LightGBM surface.

## Contents

1. Setup and configuration
2. Exp072 full replay cache and self-GR source check
3. Lateral self-GR match feature ablation
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from lateral_self_gr_match_pseudotail_probe import (
    FULL_REPLAY_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    find_artifact,
    run_lateral_self_gr_match_pseudotail_probe,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "audit.mode"))
print("Parent:", cfg_get(config, "lineage.parent"))
print("Cache parent:", cfg_get(config, "lineage.cache_parent"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))
print("Active modes:", cfg_get(config, "model.training.active_modes"))
print("Active variants:", [v["name"] for v in cfg_get(config, "model.feature_ablation.active_variants", [])])
print("Self-GR feature groups:", list(cfg_get(config, "model.self_gr_match.feature_groups", {}).keys()))


## 2. Exp072 full replay cache and self-GR source check

In [ ]:
cache_path = find_artifact(
    FULL_REPLAY_TRAIN_FEATURES,
    cfg_get(config, "data.exp072_train_feature_cache_local"),
)
print("exp072 full replay train cache:", cache_path)
preview = pd.read_csv(cache_path, nrows=5, dtype={"id": str, "well": str})
print("Columns:", len(preview.columns))
preview_cols = [
    c
    for c in [
        "id",
        "well",
        "target",
        "last_known_tvt",
        "md_since",
        "eval_len",
    ]
    if c in preview.columns
]
display(preview[preview_cols])


## 3. Lateral self-GR match feature ablation

In [ ]:
summary = run_lateral_self_gr_match_pseudotail_probe(
    output_dir=paths.artifacts_dir,
    train_dir=paths.train_data_dir,
    cache_path=cfg_get(config, "data.exp072_train_feature_cache_local"),
    self_gr_config=cfg_get(config, "model.self_gr_match", {}),
    sample_weight_config=cfg_get(config, "model.sample_weight", {}),
    variants=cfg_get(config, "model.feature_ablation.active_variants", []),
    modes=cfg_get(config, "model.training.modes", {}),
    active_modes=cfg_get(config, "model.training.active_modes", []),
    n_splits=int(cfg_get(config, "validation.n_folds", 5)),
    fast=bool(cfg_get(config, "audit.fast", False)),
    early_stopping_rounds=int(cfg_get(config, "model.training.early_stopping_rounds", 250)),
    max_rows=cfg_get(config, "model.training.max_rows"),
    max_train_rows=cfg_get(config, "model.training.max_train_rows"),
    save_models=bool(cfg_get(config, "model.training.save_models", True)),
    save_predictions=bool(cfg_get(config, "model.training.save_predictions", True)),
    top_n_importance=int(cfg_get(config, "model.training.top_n_importance", 40)),
)
print(json.dumps(summary, indent=2)[:4000])


## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_metrics.csv")
by_well = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_by_well.csv")
bucket_metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_bucket_metrics.csv")
self_gr_summary = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_self_gr_feature_summary.csv")
sample_weight_summary = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_sample_weight_summary.csv")
importance_mean = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_importance_mean.csv")
manifest_path = paths.artifacts_dir / f"{OUTPUT_PREFIX}_lgb_models" / "manifest.json"

pooled = metrics[metrics["fold"].astype(str).eq("pooled")].sort_values("rmse_tvt")
display(pooled)
display(self_gr_summary)
display(sample_weight_summary)
display(bucket_metrics.head(40))
display(by_well.head(30))
display(importance_mean.head(50))
print("Model manifest:", manifest_path, "exists=", manifest_path.exists())
print("Feature importance plot:", paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_importance_mean_top.png")
